# SepsisGuard-UAR — Week 2 preprocessing, CPU

Requires a completed Week 1 run. Fit medians, missing/constant-feature filtering, and scaling using **only A training patients**. Use the original target at the endpoint of each trailing 12-hour window. Forward fill resets at patient boundaries; left padding has its own time mask. B remains locked. No training jobs, GPU use, or clinical claims are part of this notebook.

If Colab reset, choose the exact prior Week 1 run ID below. Never silently replace the split manifest or choose it based on model scores.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, subprocess, shutil, json, uuid
LOCAL = os.environ.get('SEPSISGUARD_LOCAL_VALIDATION') == '1'
REPO_URL = 'https://github.com/shakiful/Sepsisguard-Uar.git'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
if LOCAL:
    PROJECT = Path(os.environ.get('SEPSISGUARD_PROJECT', os.getcwd())).resolve()
    DRIVE_ROOT = PROJECT/'runs/local_checkpoints'
    print('LOCAL FALLBACK: no Colab execution or Drive write is claimed.')
else:
    PROJECT = Path('/content/sepsisguard-uar')
    if PROJECT.exists():
        assert (PROJECT/'.git').is_dir(), 'Existing folder is not a Git checkout; preserve it and choose a new project path.'
        remote = subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip()
        assert remote.lower().rstrip('/').removesuffix('.git') == REPO_URL.lower().removesuffix('.git'), 'Unexpected repository; stop before touching existing work.'
    else:
        subprocess.run(['git','clone','--branch','main','--single-branch',REPO_URL,str(PROJECT)],check=True)
    # Inspect the allocated runtime before requesting Drive access.
    preliminary = subprocess.check_output([sys.executable,str(PROJECT/'scripts/compute_check.py')],text=True)
    print(preliminary)
    assert not json.loads(preliminary)['cuda_available'], 'Select CPU runtime for data preparation.'
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/SepsisGuard-UAR')
os.chdir(PROJECT)
SOURCE_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
for folder in ('starter','dataset-archives','reports','models','runs'):
    (DRIVE_ROOT/folder).mkdir(parents=True,exist_ok=True)
RUNS = PROJECT/'runs'/RUN_ID
RUNS.mkdir(parents=True,exist_ok=True)
(RUNS/'source.json').write_text(json.dumps({'repository':REPO_URL,'commit':SOURCE_COMMIT,'run_id':RUN_ID,'environment':'local_fallback' if LOCAL else 'google_colab'},indent=2)+'\n')
def run(command,name):
    print('RUN:',name,flush=True)
    r=subprocess.run(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    (RUNS/(name+'.log')).write_text(r.stdout)
    print(r.stdout)
    r.check_returncode()
    return r
DATA = PROJECT/'data/raw'
CACHE = DRIVE_ROOT/'dataset-archives'
print('Run ID:',RUN_ID,'Source commit:',SOURCE_COMMIT)


In [ ]:
run([sys.executable,'-m','pip','install','-q','-e','.[dev,download,torch]'],'install')
run([sys.executable,'scripts/compute_check.py','--output',RUNS/'compute.json'],'compute')
assert not json.loads((RUNS/'compute.json').read_text())['cuda_available'], 'Use CPU for preprocessing'
run([sys.executable,'-m','pip','freeze'],'environment')

## Restore the exact Week 1 manifest and data if necessary

In [ ]:
WEEK1_RUN_ID = ''  # Set explicitly after a Colab reset; leave blank in the same runtime.
manifest = PROJECT/'reports/week1/split_manifest.csv'
if WEEK1_RUN_ID:
    saved = DRIVE_ROOT/'reports'/WEEK1_RUN_ID/'week1'
    assert (saved/'DATA_AUDIT.json').is_file(), 'A completed, audited Week 1 run is required'
    shutil.copytree(saved, PROJECT/'reports/week1', dirs_exist_ok=True)
assert manifest.exists(), 'Complete Week 1 or set WEEK1_RUN_ID to its saved run ID'
assert (PROJECT/'reports/week1/DATA_AUDIT.json').exists()
run([sys.executable,'-m','sepsisguard.download','--data-root','data/raw',
     '--archives-dir',DRIVE_ROOT/'dataset-archives'],'restore-data')
previous_audit = json.loads((PROJECT/'reports/week1/DATA_AUDIT.json').read_text())
# Revalidate fingerprints and manifest before fitting.
run([sys.executable,'-m','sepsisguard.audit','--data-root','data/raw','--output-dir','reports/week1',
     '--manifest',manifest],'audit')
current_audit = json.loads((PROJECT/'reports/week1/DATA_AUDIT.json').read_text())
assert current_audit['cohort_sha256'] == previous_audit['cohort_sha256'], 'Dataset changed since Week 1; stop and investigate'


## Train-only fit, causal windows, deterministic DataLoader checks

In [ ]:
run([sys.executable,'-m','pytest','-q'],'pytest')
run([sys.executable,'-m','scripts.run_week2','--data-root','data/raw','--manifest',manifest,
     '--output','reports/week2','--min-observed-fraction','0.0'],'preprocessing')
result=json.loads(Path('reports/week2/WEEK2_RESULT.json').read_text())
assert result['test_B_opened'] is False
print(json.dumps(result,indent=2))

## Save the fitted artifact and provenance to private Drive

In [ ]:
shutil.copytree(PROJECT/'reports/week2',DRIVE_ROOT/'reports'/RUN_ID/'week2',dirs_exist_ok=True)
shutil.copytree(PROJECT/'reports/week2',DRIVE_ROOT/'models'/RUN_ID/'preprocessing',dirs_exist_ok=True)
(RUNS/'execution.json').write_text(json.dumps({'run_id':RUN_ID,'environment':'local_fallback' if LOCAL else 'google_colab','drive_saved':not LOCAL},indent=2))
shutil.copytree(RUNS,DRIVE_ROOT/'runs'/RUN_ID,dirs_exist_ok=True)
print('Saved preprocessing milestone:',DRIVE_ROOT/'reports'/RUN_ID/'week2')
print('Do not tune preprocessing or models using B. Supervised feature selection, if added later, belongs inside A/train patient folds.')